In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Define model name
model_name = "microsoft/phi-2"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True if device == "cuda" else False,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# Apply LoRA
peft_model = get_peft_model(model, lora_config)

In [ ]:
# Load new dataset
dataset = load_dataset("yelp_review_full", split="train[:40]")

README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/299M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
def format_instruction(text):
    return f"Summarize the restaurant review: {text}\nOutput:"

In [ ]:
def tokenize_function(examples):
    formatted_texts = [format_instruction(text) for text in examples["text"]]
    return tokenizer(formatted_texts, padding="max_length", truncation=True, max_length=256, return_tensors="pt")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./phi_lora_summary",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train the model
trainer.train()

Step,Training Loss
5,3.315000
10,3.302100
15,3.166100
20,3.153200
25,3.090600
30,3.113600
35,3.129400
40,2.997800
45,2.947100
50,3.178900


TrainOutput(global_step=50, training_loss=3.139371700286865, metrics={'train_runtime': 86.3064, 'train_samples_per_second': 2.317, 'train_steps_per_second': 0.579, 'total_flos': 814257537024000.0, 'train_loss': 3.139371700286865, 'epoch': 5.0})

In [ ]:
# Save LoRA adapter
peft_model.save_pretrained("./phi_lora_adapters")
print("LoRA adapters saved.")

LoRA adapters saved.


In [1]:
import torch
from transformers import AutoTokenizer
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Define model name
model_name = "microsoft/phi-2"

# Load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load the fine-tuned model with LoRA adapters
base_model = AutoModelForCausalLM.from_pretrained(model_name)
peft_model = PeftModel.from_pretrained(base_model, "./phi_lora_adapters")  # Load LoRA adapter

peft_model.eval()  # Set to evaluation mode

# Function to test the model
def generate_summary(review_text):
    prompt = f"Summarize the restaurant review: {review_text}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=256)
    input_ids = inputs["input_ids"].to(peft_model.device)

    with torch.no_grad():
        output_ids = peft_model.generate(input_ids, max_length=50, num_return_sequences=1)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Test with sample reviews
test_reviews = [
    "The pizza was absolutely delicious! The crust was perfectly crispy, and the cheese was just right.",
    "Terrible service. We waited an hour for food, and it arrived cold. Not coming back!",
    "The ambiance was great, and the food was amazing. Highly recommend this place!"
]

# Run inference
for review in test_reviews:
    summary = generate_summary(review)
    print(f"Review: {review}\nSummary: {summary}\n{'-'*50}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Review: The pizza was absolutely delicious! The crust was perfectly crispy, and the cheese was just right.
Summary: Summarize the restaurant review: The pizza was absolutely delicious! The crust was perfectly crispy, and the cheese was just right.
Output: The pizza was incredibly tasty, with a perfectly crispy crust and perfectly melted cheese.

--------------------------------------------------


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Review: Terrible service. We waited an hour for food, and it arrived cold. Not coming back!
Summary: Summarize the restaurant review: Terrible service. We waited an hour for food, and it arrived cold. Not coming back!
Output: The restaurant had terrible service, with a long wait time and cold food. The reviewer decided not to
--------------------------------------------------
Review: The ambiance was great, and the food was amazing. Highly recommend this place!
Summary: Summarize the restaurant review: The ambiance was great, and the food was amazing. Highly recommend this place!
Output: The restaurant review praises the great ambiance and amazing food, highly recommending the place.

--------------------------------------------------
